In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import contextlib
import io
import json
import random
import sys

import numpy as np
import pandas as pd

def resolve_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'fbn_pipeline').exists():
            return candidate
    raise RuntimeError('Project root not found')

PROJECT_ROOT = resolve_project_root()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from fbn_pipeline import (
    DevelopmentSelection,
    ExperimentConfig,
    apply_preset,
    load_training_split,
    lock_development_config,
    run_development,
    run_final_evaluation_from_lock,
    select_best_development_config,
)
from fbn_pipeline.metrics import per_subject_metrics

SEED = 42
K_FOLDS = 5
PRIMARY_SELECTION_METRIC = 'oof_balanced_accuracy'
TIE_BREAK_RULES = [
    'oof_macro_f1',
    'oof_auc',
    'lower_model_complexity',
    'configuration_id_lexicographic',
]
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'publication_protocol_v2'
DEVELOPMENT_DIR = ARTIFACT_ROOT / 'development'
DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = DEVELOPMENT_DIR / 'publication_candidate_manifest.json'
FTS_CSV_PATH = DEVELOPMENT_DIR / 'fts_selection_oof.csv'
RULE_CSV_PATH = DEVELOPMENT_DIR / 'anfis_rule_selection_oof.csv'
BN_CSV_PATH = DEVELOPMENT_DIR / 'bn_selection_oof.csv'
ABLATION_CSV_PATH = DEVELOPMENT_DIR / 'pipeline_ablation_oof.csv'
FINAL_OOF_PATH = DEVELOPMENT_DIR / 'final_selected_oof_predictions.npz'
SUMMARY_JSON_PATH = DEVELOPMENT_DIR / 'publication_development_summary.json'
SUMMARY_MD_PATH = DEVELOPMENT_DIR / 'publication_development_summary.md'
LOCK_PATH = ARTIFACT_ROOT / 'locked_config.json'
random.seed(SEED)
np.random.seed(SEED)

def stable_hash(obj) -> str:
    payload = json.dumps(obj, sort_keys=True, separators=(',', ':'), default=float)
    import hashlib
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def timestamp_utc() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def cfg_to_stage_dict(cfg: ExperimentConfig) -> dict:
    return cfg.to_dict()

def run_candidate_set(candidate_items, X_train, y_train, user_train):
    results = []
    for candidate_id, cfg in candidate_items:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            result = run_development(
                X_train,
                y_train,
                user_train,
                cfg,
                configuration_id=candidate_id,
                print_progress=False,
            )
        results.append(result)
    selection = select_best_development_config(results)
    return results, selection

def complexity_tuple(cfg: ExperimentConfig):
    return (int(cfg.n_rules), int(cfg.n_sets), int(cfg.n_bins), cfg.variant_name or '')

def fold_id_array(fold_indices, n_rows: int) -> np.ndarray:
    fold_ids = np.full(n_rows, -1, dtype=int)
    for fold_id, (_, va_idx) in enumerate(fold_indices):
        fold_ids[va_idx] = fold_id
    if np.any(fold_ids < 0):
        raise ValueError('Some rows were not assigned a fold id')
    return fold_ids

def fold_summary_table(fold_indices, y_train, user_train) -> pd.DataFrame:
    rows = []
    for fold_id, (tr_idx, va_idx) in enumerate(fold_indices, start=1):
        rows.append({
            'fold': fold_id,
            'train_n': int(len(tr_idx)),
            'valid_n': int(len(va_idx)),
            'train_rest': int(np.sum(y_train[tr_idx] == 0)),
            'train_task': int(np.sum(y_train[tr_idx] == 1)),
            'valid_rest': int(np.sum(y_train[va_idx] == 0)),
            'valid_task': int(np.sum(y_train[va_idx] == 1)),
            'train_subjects': ','.join(sorted(np.unique(user_train[tr_idx]).tolist())),
            'valid_subjects': ','.join(sorted(np.unique(user_train[va_idx]).tolist())),
        })
    return pd.DataFrame(rows)

def stage_table(results, selection, stage_name: str, parameter_builder, context_builder=None) -> pd.DataFrame:
    rows = []
    for rank, result in enumerate(selection.ranked_results, start=1):
        row = {
            'rank': rank,
            'configuration_id': result.configuration_id,
            'oof_accuracy': result.oof_metrics['accuracy'],
            'oof_balanced_accuracy': result.oof_metrics['bal_acc'],
            'oof_precision': result.oof_metrics['precision'],
            'oof_recall': result.oof_metrics['recall'],
            'oof_task_f1': result.oof_metrics['f1'],
            'oof_macro_f1': result.oof_metrics['macro_f1'],
            'oof_auc': result.oof_metrics['auc'],
            'selected_oof_threshold': result.threshold,
            'selected': result.configuration_id == selection.selected_result.configuration_id,
            'selection_metric': selection.ranking_metric,
        }
        row.update(parameter_builder(result))
        if context_builder is not None:
            row.update(context_builder(result))
        rows.append(row)
    df = pd.DataFrame(rows)
    if any(col.startswith('test_') for col in df.columns):
        raise ValueError(f'{stage_name} table contains forbidden test columns')
    return df

def print_df(title: str, df: pd.DataFrame) -> None:
    print(title)
    print(df.to_string(index=False))

BN_VARIANTS = {
    'BN_SOFT_NAIVE_BIN5': {
        'bn_structure': 'naive',
        'bn_soft': True,
        'n_bins': 5,
        'alpha': 0.5,
        'bn_reliability': False,
        'calibrate': 'none',
        'notes': 'existing A2_soft publication BN candidate',
    },
    'BN_SOFT_NAIVE_BIN3': {
        'bn_structure': 'naive',
        'bn_soft': True,
        'n_bins': 3,
        'alpha': 0.5,
        'bn_reliability': False,
        'calibrate': 'none',
        'notes': 'existing A2_soft_bin3 publication BN candidate',
    },
    'BN_SOFT_TAN_BIN5': {
        'bn_structure': 'tan',
        'bn_soft': True,
        'n_bins': 5,
        'alpha': 0.5,
        'bn_reliability': False,
        'calibrate': 'none',
        'notes': 'existing A2_soft_tan publication BN candidate',
    },
    'BN_HARD_TAN_BIN5': {
        'bn_structure': 'tan',
        'bn_soft': False,
        'n_bins': 5,
        'alpha': 0.5,
        'bn_reliability': False,
        'calibrate': 'none',
        'notes': 'existing A2_hard_tan publication BN candidate',
    },
}

def make_stage_a_candidates():
    base = ExperimentConfig(data_path='last_30s_segments')
    ids = ['EXP1_MF_TRI', 'EXP1_MF_GAUSS', 'EXP1_MF_TRAP', 'EXP1_MF_GBELL']
    return [(candidate_id, apply_preset(base, candidate_id)) for candidate_id in ids]

def make_stage_b_candidates(stage_a_id: str, stage_a_cfg: ExperimentConfig):
    candidates = []
    for rule_count in [3, 6, 8, 12]:
        candidate_id = f'PUB_RULE__{stage_a_id}__R{rule_count}'
        cfg = stage_a_cfg.merged(
            n_rules=rule_count,
            experiment_group='PUB_RULES',
            variant_name=candidate_id,
        )
        candidates.append((candidate_id, cfg))
    return candidates

def make_stage_c_candidates(stage_a_id: str, stage_b_id: str, base_cfg: ExperimentConfig):
    candidates = []
    for variant_name, variant_cfg in BN_VARIANTS.items():
        candidate_id = f'PUB_BN__{stage_a_id}__{stage_b_id.split("__")[-1]}__{variant_name}'
        cfg = base_cfg.merged(
            bn_structure=variant_cfg['bn_structure'],
            bn_soft=variant_cfg['bn_soft'],
            n_bins=variant_cfg['n_bins'],
            alpha=variant_cfg['alpha'],
            bn_reliability=variant_cfg['bn_reliability'],
            calibrate=variant_cfg['calibrate'],
            experiment_group='PUB_BN',
            variant_name=candidate_id,
        )
        candidates.append((candidate_id, cfg))
    return candidates

def make_stage_d_candidates(final_cfg: ExperimentConfig):
    return [
        ('ABLATION_FULL', final_cfg.merged(ablation='exp1', variant_name='ABLATION_FULL', experiment_group='PUB_ABLATION')),
        ('ABLATION_NO_BN', final_cfg.merged(ablation='exp2', fusion_mode_when_no_bn='mean', variant_name='ABLATION_NO_BN', experiment_group='PUB_ABLATION')),
        ('ABLATION_NO_ANFIS', final_cfg.merged(ablation='exp3', fts_summary_mode='diag_high', variant_name='ABLATION_NO_ANFIS', experiment_group='PUB_ABLATION')),
        ('ABLATION_NO_FTS', final_cfg.merged(ablation='exp4', no_stage1_feature_mode='stats', variant_name='ABLATION_NO_FTS', experiment_group='PUB_ABLATION')),
    ]

def candidate_manifest_payload():
    stage_a = {candidate_id: cfg_to_stage_dict(cfg) for candidate_id, cfg in make_stage_a_candidates()}
    stage_b = {}
    stage_c = {}
    for stage_a_id, stage_a_cfg in make_stage_a_candidates():
        stage_b_branch = {}
        for stage_b_id, stage_b_cfg in make_stage_b_candidates(stage_a_id, stage_a_cfg):
            stage_b_branch[stage_b_id] = cfg_to_stage_dict(stage_b_cfg)
            stage_c_branch = stage_c.setdefault(stage_a_id, {})
            stage_c_branch[stage_b_id] = {
                candidate_id: cfg_to_stage_dict(cfg)
                for candidate_id, cfg in make_stage_c_candidates(stage_a_id, stage_b_id, stage_b_cfg)
            }
        stage_b[stage_a_id] = stage_b_branch
    payload = {
        'protocol_version': 'publication_protocol_v2',
        'manifest_schema_version': 1,
        'created_at_utc': timestamp_utc(),
        'seed': SEED,
        'k_folds': K_FOLDS,
        'grouping_variable': 'user_train',
        'grouping_protocol': 'GroupKFold grouped by training subject IDs only',
        'primary_selection_metric': PRIMARY_SELECTION_METRIC,
        'tie_break_rules': TIE_BREAK_RULES,
        'frozen_before_execution': True,
        'selection_uses_heldout_metrics': False,
        'selection_uses_heldout_predictions': False,
        'manifest_freeze_statement': 'Manifest frozen before the first development candidate was executed.',
        'test_usage_statement': 'No test metrics are used anywhere in publication development.',
        'stage_a_candidates': stage_a,
        'stage_a_common_context': {
            key: value
            for key, value in cfg_to_stage_dict(apply_preset(ExperimentConfig(data_path='last_30s_segments'), 'THESIS_FINAL')).items()
            if key not in {'mf_type', 'variant_name', 'experiment_group'}
        },
        'stage_b_candidates_by_stage_a': stage_b,
        'stage_c_candidates_by_stage_a_and_stage_b': stage_c,
        'stage_d_ablation_templates': {
            'ABLATION_FULL': {'ablation': 'exp1'},
            'ABLATION_NO_BN': {'ablation': 'exp2', 'fusion_mode_when_no_bn': 'mean'},
            'ABLATION_NO_ANFIS': {'ablation': 'exp3', 'fts_summary_mode': 'diag_high'},
            'ABLATION_NO_FTS': {'ablation': 'exp4', 'no_stage1_feature_mode': 'stats'},
        },
        'stage_c_exclusions': [
            'No explicit hard-evidence naive-BN publication preset exists, so it was not added post hoc.',
            'Calibration variants were excluded from Stage C because the current calibration implementation fits calibrators on in-fold training probabilities and is not a nested/OOF-safe publication selection procedure.',
            'Reliability weighting was excluded from Stage C because the existing A2_plus candidate couples reliability weighting with isotonic calibration and robust/scaled FTS changes rather than isolating BN fusion alone.',
            'No clean isolated smoothing-alpha publication candidate exists beyond the current 0.5 setting, so alpha was not expanded into a new post hoc grid.',
        ],
    }
    payload['candidate_counts'] = {
        'stage_a_executed': 4,
        'stage_b_branch_size': 4,
        'stage_c_branch_size': 4,
        'stage_d_executed': 4,
        'manifest_stage_b_concrete_total': sum(len(branch) for branch in stage_b.values()),
        'manifest_stage_c_concrete_total': sum(len(branch_candidates) for branch in stage_c.values() for branch_candidates in branch.values()),
    }
    payload['all_candidate_ids'] = sorted(
        list(stage_a.keys())
        + [candidate_id for branch in stage_b.values() for candidate_id in branch.keys()]
        + [candidate_id for branch in stage_c.values() for branch_candidates in branch.values() for candidate_id in branch_candidates.keys()]
        + ['ABLATION_FULL', 'ABLATION_NO_BN', 'ABLATION_NO_ANFIS', 'ABLATION_NO_FTS']
    )
    payload['manifest_hash'] = stable_hash({k: v for k, v in payload.items() if k != 'manifest_hash'})
    return payload

PREEXISTING_FINAL_ARTIFACTS = {}
if ARTIFACT_ROOT.exists():
    for path in ARTIFACT_ROOT.rglob('*'):
        if path.is_file() and 'final_test' in path.name:
            PREEXISTING_FINAL_ARTIFACTS[str(path)] = path.stat().st_mtime

final_evaluation_executed = False
print('PROTOCOL MODE = PUBLICATION_DEVELOPMENT_ONLY')
print('FINAL TEST ACCESS = DISABLED')
print(f'project_root = {PROJECT_ROOT}')
print(f'seed = {SEED}')
print(f'fold_count = {K_FOLDS}')
print(f'primary_ranking_metric = {PRIMARY_SELECTION_METRIC}')
print(f'tie_break_rules = {TIE_BREAK_RULES}')
print(f'development_artifact_directory = {DEVELOPMENT_DIR}')


PROTOCOL MODE = PUBLICATION_DEVELOPMENT_ONLY
FINAL TEST ACCESS = DISABLED
project_root = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final
seed = 42
fold_count = 5
primary_ranking_metric = oof_balanced_accuracy
tie_break_rules = ['oof_macro_f1', 'oof_auc', 'lower_model_complexity', 'configuration_id_lexicographic']
development_artifact_directory = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final\artifacts\publication_protocol_v2\development


In [2]:
training_split = load_training_split(PROJECT_ROOT / 'last_30s_segments')
X_train = training_split.X_train
y_train = training_split.y_train
user_train = training_split.user_train
label_counts = {
    'rest': int(np.sum(y_train == 0)),
    'task': int(np.sum(y_train == 1)),
}
training_subject_ids = sorted(np.unique(user_train).tolist())
print(f'training_shape = {X_train.shape}')
print(f'training_label_counts = {label_counts}')
print(f'training_subject_ids = {training_subject_ids}')


training_shape = (632, 4, 30)
training_label_counts = {'rest': 317, 'task': 315}
training_subject_ids = ['1mpau', '2nxs5', '5gpsc', '7swyk', '8a1ep', 'b7mrd', 'c24ur', 'dkhty', 'e4gay', 'ef5rq', 'f3j25', 'hpbxa', 'ibvx8', 'iz2ps', 'rc1in', 'tn4vl', 'wjxci', 'yljm5']


In [3]:
manifest = candidate_manifest_payload()
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
print('candidate_groups = [stage_a, stage_b, stage_c, stage_d_templates]')
print(f"stage_a_ids = {sorted(manifest['stage_a_candidates'].keys())}")
print(f"stage_b_branch_size = {manifest['candidate_counts']['stage_b_branch_size']}")
print(f"stage_c_branch_size = {manifest['candidate_counts']['stage_c_branch_size']}")
print(f"stage_d_template_ids = {sorted(manifest['stage_d_ablation_templates'].keys())}")
print(f'manifest_path = {MANIFEST_PATH}')
print(f"manifest_hash = {manifest['manifest_hash']}")


candidate_groups = [stage_a, stage_b, stage_c, stage_d_templates]
stage_a_ids = ['EXP1_MF_GAUSS', 'EXP1_MF_GBELL', 'EXP1_MF_TRAP', 'EXP1_MF_TRI']
stage_b_branch_size = 4
stage_c_branch_size = 4
stage_d_template_ids = ['ABLATION_FULL', 'ABLATION_NO_ANFIS', 'ABLATION_NO_BN', 'ABLATION_NO_FTS']
manifest_path = C:\Users\doqua\Desktop\fuzzy-bayesian-cognitive-load\fts-anfis-bns-final\artifacts\publication_protocol_v2\development\publication_candidate_manifest.json
manifest_hash = a05d840c75ab97438ec611c72b78b0822553ee626f122e4de3ceec64940c3efe


In [4]:
stage_a_candidates = make_stage_a_candidates()
stage_a_results, stage_a_selection = run_candidate_set(stage_a_candidates, X_train, y_train, user_train)
stage_a_df = stage_table(
    stage_a_results,
    stage_a_selection,
    'stage_a',
    parameter_builder=lambda result: {
        'fts_parameters': json.dumps({
            'n_sets': result.config.n_sets,
            'mf_type': result.config.mf_type,
            'mf_sigma_scale': result.config.mf_sigma_scale,
            'gbell_b': result.config.gbell_b,
            'fts_norm': result.config.fts_norm,
            'trap_plateau_scale': result.config.trap_plateau_scale,
            'trap_width_scale': result.config.trap_width_scale,
            'grid_margin_scale': result.config.grid_margin_scale,
        }, sort_keys=True),
    },
    context_builder=lambda result: {
        'fixed_anfis_bn_context': json.dumps({
            'anfis_mf_type': result.config.anfis_mf_type,
            'n_rules': result.config.n_rules,
            'sigma_scale': result.config.sigma_scale,
            'ridge_lambda': result.config.ridge_lambda,
            'n_epochs': result.config.n_epochs,
            'lr': result.config.lr,
            'n_bins': result.config.n_bins,
            'alpha': result.config.alpha,
            'bn_structure': result.config.bn_structure,
            'bn_soft': result.config.bn_soft,
        }, sort_keys=True),
    },
)
stage_a_df.to_csv(FTS_CSV_PATH, index=False)
print_df('Stage A - FTS selection (OOF only)', stage_a_df)
print(f"selected_fts_configuration = {stage_a_selection.selected_result.configuration_id}")


Stage A - FTS selection (OOF only)
 rank configuration_id  oof_accuracy  oof_balanced_accuracy  oof_precision  oof_recall  oof_task_f1  oof_macro_f1  oof_auc  selected_oof_threshold  selected      selection_metric                                                                                                                                                                   fts_parameters                                                                                                                                                                  fixed_anfis_bn_context
    1    EXP1_MF_GAUSS        0.6234                 0.6236         0.6091      0.6825       0.6437        0.6222   0.6201                   0.550      True oof_balanced_accuracy   {"fts_norm": "row", "gbell_b": 2.0, "grid_margin_scale": 0.05, "mf_sigma_scale": 0.4, "mf_type": "gaussian", "n_sets": 3, "trap_plateau_scale": 0.15, "trap_width_scale": 1.0} {"alpha": 0.5, "anfis_mf_type": "gaussian", "bn_soft": true, "bn_stru

In [5]:
selected_stage_a_result = stage_a_selection.selected_result
stage_b_candidates = make_stage_b_candidates(
    selected_stage_a_result.configuration_id,
    selected_stage_a_result.config,
)
stage_b_results, stage_b_selection = run_candidate_set(stage_b_candidates, X_train, y_train, user_train)
stage_b_df = stage_table(
    stage_b_results,
    stage_b_selection,
    'stage_b',
    parameter_builder=lambda result: {
        'rule_count': int(result.config.n_rules),
        'fts_context': json.dumps({
            'selected_stage_a_id': selected_stage_a_result.configuration_id,
            'mf_type': result.config.mf_type,
            'n_sets': result.config.n_sets,
            'fts_norm': result.config.fts_norm,
        }, sort_keys=True),
        'fixed_bn_context': json.dumps({
            'n_bins': result.config.n_bins,
            'alpha': result.config.alpha,
            'bn_structure': result.config.bn_structure,
            'bn_soft': result.config.bn_soft,
        }, sort_keys=True),
    },
)
stage_b_df.to_csv(RULE_CSV_PATH, index=False)
print_df('Stage B - ANFIS rule-count selection (OOF only)', stage_b_df)
print(f"selected_rule_configuration = {stage_b_selection.selected_result.configuration_id}")


Stage B - ANFIS rule-count selection (OOF only)
 rank             configuration_id  oof_accuracy  oof_balanced_accuracy  oof_precision  oof_recall  oof_task_f1  oof_macro_f1  oof_auc  selected_oof_threshold  selected      selection_metric  rule_count                                                                                     fts_context                                                      fixed_bn_context
    1  PUB_RULE__EXP1_MF_GAUSS__R6        0.6234                 0.6236         0.6091      0.6825       0.6437        0.6222   0.6201                   0.550      True oof_balanced_accuracy           6 {"fts_norm": "row", "mf_type": "gaussian", "n_sets": 3, "selected_stage_a_id": "EXP1_MF_GAUSS"} {"alpha": 0.5, "bn_soft": true, "bn_structure": "naive", "n_bins": 5}
    2  PUB_RULE__EXP1_MF_GAUSS__R3        0.6044                 0.6047         0.5867      0.6984       0.6377        0.6011   0.6178                   0.545     False oof_balanced_accuracy           3 {"fts_norm"

In [6]:
selected_stage_b_result = stage_b_selection.selected_result
stage_c_candidates = make_stage_c_candidates(
    selected_stage_a_result.configuration_id,
    selected_stage_b_result.configuration_id,
    selected_stage_b_result.config,
)
stage_c_results, stage_c_selection = run_candidate_set(stage_c_candidates, X_train, y_train, user_train)
stage_c_df = stage_table(
    stage_c_results,
    stage_c_selection,
    'stage_c',
    parameter_builder=lambda result: {
        'bn_parameters': json.dumps({
            'bn_structure': result.config.bn_structure,
            'bn_soft': result.config.bn_soft,
            'n_bins': result.config.n_bins,
            'alpha': result.config.alpha,
            'bn_reliability': result.config.bn_reliability,
            'calibrate': result.config.calibrate,
        }, sort_keys=True),
        'upstream_context': json.dumps({
            'selected_stage_a_id': selected_stage_a_result.configuration_id,
            'selected_stage_b_id': selected_stage_b_result.configuration_id,
            'mf_type': result.config.mf_type,
            'n_rules': result.config.n_rules,
        }, sort_keys=True),
    },
)
stage_c_df.to_csv(BN_CSV_PATH, index=False)
print('Stage C exclusions:')
for note in manifest['stage_c_exclusions']:
    print(f'- {note}')
print_df('Stage C - BN selection (OOF only)', stage_c_df)
print(f"selected_bn_configuration = {stage_c_selection.selected_result.configuration_id}")


Stage C exclusions:
- No explicit hard-evidence naive-BN publication preset exists, so it was not added post hoc.
- Calibration variants were excluded from Stage C because the current calibration implementation fits calibrators on in-fold training probabilities and is not a nested/OOF-safe publication selection procedure.
- Reliability weighting was excluded from Stage C because the existing A2_plus candidate couples reliability weighting with isotonic calibration and robust/scaled FTS changes rather than isolating BN fusion alone.
- No clean isolated smoothing-alpha publication candidate exists beyond the current 0.5 setting, so alpha was not expanded into a new post hoc grid.
Stage C - BN selection (OOF only)
 rank                              configuration_id  oof_accuracy  oof_balanced_accuracy  oof_precision  oof_recall  oof_task_f1  oof_macro_f1  oof_auc  selected_oof_threshold  selected      selection_metric                                                                        

In [7]:
selected_stage_c_result = stage_c_selection.selected_result
final_full_cfg = selected_stage_c_result.config.merged(
    ablation='exp1',
    experiment_group='PUB_FINAL',
    variant_name=selected_stage_c_result.configuration_id.replace('PUB_BN', 'PUB_FINAL', 1),
)
stage_d_candidates = make_stage_d_candidates(final_full_cfg)
stage_d_results, stage_d_selection = run_candidate_set(stage_d_candidates, X_train, y_train, user_train)
stage_d_df = stage_table(
    stage_d_results,
    stage_d_selection,
    'stage_d',
    parameter_builder=lambda result: {
        'ablation': result.config.ablation,
        'selected_full_model_context': json.dumps({
            'final_full_configuration_id': final_full_cfg.variant_name,
            'selected_stage_a_id': selected_stage_a_result.configuration_id,
            'selected_stage_b_id': selected_stage_b_result.configuration_id,
            'selected_stage_c_id': selected_stage_c_result.configuration_id,
        }, sort_keys=True),
    },
)
stage_d_df.to_csv(ABLATION_CSV_PATH, index=False)
print_df('Stage D - pipeline ablations (OOF only)', stage_d_df)
print('Ablations did not alter the previously selected full-model hyperparameters.')


Stage D - pipeline ablations (OOF only)
 rank  configuration_id  oof_accuracy  oof_balanced_accuracy  oof_precision  oof_recall  oof_task_f1  oof_macro_f1  oof_auc  selected_oof_threshold  selected      selection_metric ablation                                                                                                                                                                                                                               selected_full_model_context
    1     ABLATION_FULL        0.6234                 0.6236         0.6091      0.6825       0.6437        0.6222   0.6201                    0.55      True oof_balanced_accuracy     exp1 {"final_full_configuration_id": "PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5", "selected_stage_a_id": "EXP1_MF_GAUSS", "selected_stage_b_id": "PUB_RULE__EXP1_MF_GAUSS__R6", "selected_stage_c_id": "PUB_BN__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5"}
    2    ABLATION_NO_BN        0.6013                 0.6016         0.5806      0

In [8]:
final_candidate_id = final_full_cfg.variant_name
final_result, final_selection = None, None
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    final_result = run_development(
        X_train,
        y_train,
        user_train,
        final_full_cfg,
        configuration_id=final_candidate_id,
        print_progress=False,
    )
final_selection = select_best_development_config([final_result])
final_fold_ids = fold_id_array(final_result.fold_indices, len(final_result.oof_y))
np.savez(
    FINAL_OOF_PATH,
    y_true=final_result.oof_y,
    y_prob=final_result.oof_post,
    y_pred=final_result.oof_pred,
    subject_ids=user_train,
    fold_ids=final_fold_ids,
)
final_fold_summary_df = fold_summary_table(final_result.fold_indices, y_train, user_train)
final_subject_summary_df = pd.DataFrame(
    per_subject_metrics(
        final_result.oof_y,
        final_result.oof_pred,
        final_result.oof_post,
        user_train,
    )
)
final_config_hash = stable_hash(final_result.config.to_dict())
print(f'final_selected_configuration_id = {final_candidate_id}')
print(json.dumps(final_result.oof_metrics, indent=2))
print(f'selected_oof_threshold = {final_result.threshold}')
print(f'configuration_hash = {final_config_hash}')
print_df('Final selected fold summary', final_fold_summary_df)
print_df('Final selected per-subject OOF summary', final_subject_summary_df)


final_selected_configuration_id = PUB_FINAL__EXP1_MF_GAUSS__R6__BN_SOFT_NAIVE_BIN5
{
  "accuracy": 0.6234,
  "precision": 0.6091,
  "recall": 0.6825,
  "f1": 0.6437,
  "auc": 0.6201,
  "acc": 0.6234,
  "bal_acc": 0.6236,
  "macro_f1": 0.6222
}
selected_oof_threshold = 0.55
configuration_hash = ad2f91375057dbe40890c902ff39bdaf8c32a3b31a588f745e2aed5eba82706e
Final selected fold summary
 fold  train_n  valid_n  train_rest  train_task  valid_rest  valid_task                                                                            train_subjects          valid_subjects
    1      504      128         251         253          66          62       1mpau,2nxs5,7swyk,8a1ep,b7mrd,c24ur,dkhty,e4gay,ef5rq,f3j25,rc1in,tn4vl,wjxci,yljm5 5gpsc,hpbxa,ibvx8,iz2ps
    2      517      115         260         257          57          58 2nxs5,5gpsc,7swyk,8a1ep,b7mrd,c24ur,dkhty,e4gay,ef5rq,f3j25,hpbxa,ibvx8,iz2ps,rc1in,tn4vl       1mpau,wjxci,yljm5
    3      518      114         260         258       

In [9]:
lock_payload = lock_development_config(
    final_result,
    final_selection,
    output_path=LOCK_PATH,
    extra_metadata={'candidate_manifest_hash': manifest['manifest_hash']},
)
loaded_lock = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
required_lock_fields = {
    'schema_version', 'protocol_version', 'candidate_manifest_hash', 'configuration_id', 'config',
    'seed', 'k_folds', 'grouping_protocol', 'selection_metric', 'tie_break_rules', 'threshold',
    'threshold_source', 'oof_metrics', 'config_hash', 'lock_hash', 'timestamp_utc',
    'selected_using', 'final_test_evaluated'
}
missing_lock_fields = sorted(required_lock_fields.difference(loaded_lock))
assert not missing_lock_fields, missing_lock_fields
assert loaded_lock['candidate_manifest_hash'] == manifest['manifest_hash']
assert loaded_lock['selected_using'] == 'training_oof_only'
assert loaded_lock['threshold_source'] == 'training_oof_only'
assert loaded_lock['final_test_evaluated'] is False
assert loaded_lock['configuration_id'] == final_candidate_id
assert loaded_lock['threshold'] == final_result.threshold
assert loaded_lock['oof_metrics'] == final_result.oof_metrics
assert loaded_lock['config_hash'] == stable_hash(final_result.config.to_dict())
assert loaded_lock['lock_hash'] == stable_hash({k: v for k, v in loaded_lock.items() if k != 'lock_hash'})
print(json.dumps({
    'lock_path': str(LOCK_PATH),
    'manifest_hash': loaded_lock['candidate_manifest_hash'],
    'configuration_hash': loaded_lock['config_hash'],
    'lock_hash': loaded_lock['lock_hash'],
    'selected_oof_threshold': loaded_lock['threshold'],
    'selected_using': loaded_lock['selected_using'],
    'threshold_source': loaded_lock['threshold_source'],
    'final_test_evaluated': loaded_lock['final_test_evaluated'],
}, indent=2))


{
  "lock_path": "C:\\Users\\doqua\\Desktop\\fuzzy-bayesian-cognitive-load\\fts-anfis-bns-final\\artifacts\\publication_protocol_v2\\locked_config.json",
  "manifest_hash": "a05d840c75ab97438ec611c72b78b0822553ee626f122e4de3ceec64940c3efe",
  "configuration_hash": "ad2f91375057dbe40890c902ff39bdaf8c32a3b31a588f745e2aed5eba82706e",
  "lock_hash": "a2e479262944b50f002d304a2ecb91af6228376b7440ee639ed71169cb80aa37",
  "selected_oof_threshold": 0.55,
  "selected_using": "training_oof_only",
  "threshold_source": "training_oof_only",
  "final_test_evaluated": false
}


In [10]:
generated_tables = {
    'fts': stage_a_df,
    'rules': stage_b_df,
    'bn': stage_c_df,
    'ablation': stage_d_df,
}
assert all(not any(col.startswith('test_') for col in df.columns) for df in generated_tables.values())
development_result_fields = set(final_result.as_dict().keys())
assert not any(field.startswith('test') for field in development_result_fields)
assert callable(run_final_evaluation_from_lock)
assert final_evaluation_executed is False
final_test_csv_path = ARTIFACT_ROOT / 'final_test_results.csv'
no_new_final_test_csv = not final_test_csv_path.exists()
current_final_artifacts = {}
if ARTIFACT_ROOT.exists():
    for path in ARTIFACT_ROOT.rglob('*'):
        if path.is_file() and 'final_test' in path.name:
            current_final_artifacts[str(path)] = path.stat().st_mtime
no_final_artifact_overwrite = all(
    current_final_artifacts.get(path) == mtime
    for path, mtime in PREEXISTING_FINAL_ARTIFACTS.items()
)
lock_contains_no_test_metrics = not any(key.startswith('test') for key in loaded_lock.keys())
forbidden_tokens = ['test_acc', 'test_auc', 'test_f1', 'test_metrics', 'test_pred', 'test_post']
artifact_files_to_scan = [
    MANIFEST_PATH,
    FTS_CSV_PATH,
    RULE_CSV_PATH,
    BN_CSV_PATH,
    ABLATION_CSV_PATH,
    SUMMARY_JSON_PATH,
    SUMMARY_MD_PATH,
    LOCK_PATH,
]
summary_payload = {
    'selected_fts_candidate': selected_stage_a_result.configuration_id,
    'selected_anfis_candidate': selected_stage_b_result.configuration_id,
    'selected_bn_candidate': selected_stage_c_result.configuration_id,
    'final_proposed_model_configuration_id': final_candidate_id,
    'final_oof_metrics': final_result.oof_metrics,
    'selected_threshold': final_result.threshold,
    'oof_csv_paths': {
        'fts_selection': str(FTS_CSV_PATH),
        'anfis_rule_selection': str(RULE_CSV_PATH),
        'bn_selection': str(BN_CSV_PATH),
        'pipeline_ablation': str(ABLATION_CSV_PATH),
    },
    'manifest_hash': manifest['manifest_hash'],
    'lock_path': str(LOCK_PATH),
    'final_test_executed': False,
}
SUMMARY_JSON_PATH.write_text(json.dumps(summary_payload, indent=2, sort_keys=True), encoding='utf-8')
SUMMARY_MD_PATH.write_text(
    '\n'.join([
        '# Publication Development Summary',
        '',
        f"- Selected FTS candidate: {selected_stage_a_result.configuration_id}",
        f"- Selected ANFIS candidate: {selected_stage_b_result.configuration_id}",
        f"- Selected BN candidate: {selected_stage_c_result.configuration_id}",
        f"- Final proposed-model configuration: {final_candidate_id}",
        f"- Final OOF threshold: {final_result.threshold}",
        '',
        'All candidate selection and threshold selection were performed using out-of-fold predictions from the 18 training subjects only. The five held-out test subjects were not evaluated during publication development.',
    ]),
    encoding='utf-8',
)
scan_targets = [FTS_CSV_PATH, RULE_CSV_PATH, BN_CSV_PATH, ABLATION_CSV_PATH, SUMMARY_JSON_PATH, SUMMARY_MD_PATH, LOCK_PATH]
scan_hits = {}
for path in scan_targets:
    text = path.read_text(encoding='utf-8')
    hits = [token for token in forbidden_tokens if token in text]
    if hits:
        scan_hits[str(path)] = hits
assert not scan_hits, scan_hits
safeguards = {
    'no_test_columns_in_tables': True,
    'development_result_objects_contain_no_test_fields': True,
    'final_evaluation_function_unexecuted': final_evaluation_executed is False,
    'no_new_final_test_csv_written': no_new_final_test_csv,
    'no_existing_final_test_artifact_overwritten': no_final_artifact_overwrite,
    'lock_contains_no_test_metrics': lock_contains_no_test_metrics,
    'forbidden_field_scan_hits': scan_hits,
}
print(json.dumps(safeguards, indent=2))


{
  "no_test_columns_in_tables": true,
  "development_result_objects_contain_no_test_fields": true,
  "final_evaluation_function_unexecuted": true,
  "no_new_final_test_csv_written": true,
  "no_existing_final_test_artifact_overwritten": true,
  "lock_contains_no_test_metrics": true,
  "forbidden_field_scan_hits": {}
}


In [11]:
final_summary = {
    'publication_development_completed': True,
    'candidate_manifest_frozen': True,
    'fts_selection_completed': True,
    'anfis_selection_completed': True,
    'bn_selection_completed': True,
    'ablation_oof_completed': True,
    'final_selected_oof_completed': True,
    'publication_lock_created': True,
    'publication_lock_validated': True,
    'final_test_evaluation_executed': False,
    'test_metrics_generated': False,
}
print(json.dumps(final_summary, indent=2))


{
  "publication_development_completed": true,
  "candidate_manifest_frozen": true,
  "fts_selection_completed": true,
  "anfis_selection_completed": true,
  "bn_selection_completed": true,
  "ablation_oof_completed": true,
  "final_selected_oof_completed": true,
  "publication_lock_created": true,
  "publication_lock_validated": true,
  "final_test_evaluation_executed": false,
  "test_metrics_generated": false
}
